# 10. Compare bulkmc

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
from glob import glob


In [ ]:
# seq = {}
# with gzip.open('/home/zhoujt/software/wgbs_tools/references/hg38/hg38.fa.gz', 'rt') as fin:
#     for line in fin:
#         if line[0]=='>':
#             chrom = line[1:].strip()
#             seq[chrom] = ''
#         else:
#             seq[chrom] += line.strip()


In [ ]:
# from bolero import Genome
# genome = Genome('hg38')
# seq = {}
# ref = np.array(['A','C','G','T'])
# for chrom in genome.chrom_sizes.keys():
#     tmp = genome.get_region_one_hot(f'{chrom}:0-{genome.chrom_sizes[chrom]}')
#     idx = np.where(tmp)
#     nn = np.array(['N' for i in range(tmp.shape[0])])
#     nn[idx[0]] = ref[idx[1]]
#     seq[chrom] = ''.join(nn)
#     print(chrom)



In [ ]:
ref = pd.read_csv('/home/zhoujt/software/wgbs_tools/references/hg38/CpG.bed.gz', sep='\t', header=None, index_col=2)

from pyfaidx import Fasta
f = Fasta('/large_experiments/zhoulab/ref/hg38/fasta/hg38.fa')
seq = {}
for chrom in ref[0].unique():
    seq[chrom] = f[chrom][:].seq.upper()
    print(chrom)

ref['seq'] = [seq[xx][(yy-1):(yy+2)] for xx,yy in zip(ref[0].values, ref[1].values)]
ref.to_hdf('hg38.CpG.3bp.hdf', key='data')


In [ ]:
ref = pd.read_hdf('hg38.CpG.3bp.hdf', key='data')
ref.index = ref.index - 1

beta_list = glob('raw/*.beta')
# beta_file = 'raw/GSM5652382_Small-int-Endocrine-Z00000436.hg38.beta'


In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

def beta2allc(beta_file):
    global ref
    allc_file = beta_file.replace('.hg38.beta', '.allc.tsv').replace('raw/', 'allc/')
    content = np.fromfile(beta_file, dtype=np.uint8).reshape((-1, 2))
    allc = pd.concat([ref, pd.DataFrame(content, columns=['mc', 'cov'])], axis=1)
    allc = allc.loc[allc['cov']>0]
    allc['strand'] = '+'
    allc['last'] = 1
    allc[[0, 1, 'strand', 'seq', 'mc', 'cov', 'last']].to_csv(allc_file, sep='\t', header=False, index=False)
    cmd = f'bgzip -f {allc_file}; allcools tabix-allc --reindex {allc_file}.gz'
    os.system(cmd)
    return


In [ ]:
cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for beta_file in beta_list:
        future = executor.submit(beta2allc, beta_file=beta_file)
        futures[future] = beta_file
    result = []
    for future in as_completed(futures):
        beta_file = futures[future]
        result.append(future.result())
        print(f'{beta_file} finished')


In [ ]:
!for f in $(ls *allc.tsv.gz); do (gzip -d ${f}; bgzip $(echo $f | sed 's/.gz//g'); allcools tabix-allc ${f};)  & if (( ++j % 20 == 0 )); then echo "Waiting..."; wait; fi; done


In [ ]:
!allcools generate-dataset --allc_table allclist.tsv --output_path Loyfer2023_1kb.mcds --chrom_size_path /large_experiments/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes --obs_dim cell --cpu 48 --chunk_size 1 --regions chrom1k 1000 --quantifiers chrom1k count CGN


In [ ]:
from glob import glob 
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib import cm as cm
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.mcds import MCDS

from scipy.stats import pearsonr
from sklearn.metrics import pairwise_distances
import seaborn as sns
import h5py
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/compare_bulkmc/'


In [ ]:
L2_meta = pd.read_csv(f'{indir}subtype_meta.tsv', header=0, index_col=0, sep='\t')
L2_annot = L2_meta['celltype_L2_both_abbr'].to_dict()

## Loyfer2023

In [ ]:
bulk_dir = f'{ENTEX_ROOT}/bulk_mC/Loyfer2023/'


In [ ]:
mcds = MCDS.open(f'{bulk_dir}Loyfer2023_1kb.mcds', var_dim='chrom1k')
mcds

In [ ]:
mcds = mcds.assign_coords({'chrom1k_chrom': ('chrom1k', mcds.coords['chrom1k_chrom'].data),
                    'chrom1k_start': ('chrom1k', mcds.coords['chrom1k_start'].data),
                    'chrom1k_end': ('chrom1k', mcds.coords['chrom1k_end'].data),
                   })
mcds

In [ ]:
# mcds.add_feature_cov_mean(var_dim='chrom1k', plot=False)
cov = mcds['chrom1k_da'].sel(count_type='cov').mean(dim='cell').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, binrange=(0,1200), ax=ax)

In [ ]:
mcds = mcds.sel({'chrom1k':cov.index[cov>20]})

In [ ]:
black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)

exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)

In [ ]:
mcds.add_mc_frac(normalize_per_cell=True, clip_norm_value=10)


In [ ]:
data = mcds['chrom1k_da_frac'].to_pandas()


In [ ]:
data.to_hdf(f'{bulk_dir}Loyfer2023_1kb.hdf', key='data')


In [ ]:
data = pd.read_hdf(f'{bulk_dir}Loyfer2023_1kb.hdf', key='data')


In [ ]:
from sklearn.metrics import pairwise_distances
dist = pairwise_distances(data, data, 'cosine')
dist = pd.DataFrame(dist, columns=data.index, index=data.index)
# dist.to_hdf(f'{bulk_dir}cosine_Loyfer2023.hdf', key='data')


In [ ]:
dist = pd.read_hdf(f'{bulk_dir}cosine_Loyfer2023.hdf', key='data')
cg = sns.clustermap(dist, vmax=0.02, figsize=(20,20), metric='cosine', method='single', xticklabels=2, yticklabels=2)
corder = cg.dendrogram_col.reordered_ind.copy()


In [ ]:
mcds_cluster = MCDS.open(f'{indir}merged_allc/subtype1k.mcds', var_dim='chrom1k')
mcds_cluster

In [ ]:
# group_meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', sep='\t', header=0, index_col=0)
# mcds_cluster = mcds_cluster.assign_coords(L2_any=('cell', group_meta.loc[mcds_cluster.get_index('cell'), 'L2_any']))
# mcds_cluster = mcds_cluster.groupby('L2_any').sum()
# mcds_cluster = MCDS(mcds_cluster, obs_dim='L2_any', var_dim='chrom1k')
mcds_cluster = mcds_cluster.sel({'mc_type':'CGN'})


In [ ]:
cov = mcds_cluster['chrom1k_da'].sel(count_type='cov').mean(dim='cell').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, binrange=(0,500), ax=ax)

In [ ]:
mcds_cluster = mcds_cluster.sel({'chrom1k':cov.index[cov>20]})

In [ ]:
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'
mcds_cluster = mcds_cluster.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)

exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds_cluster = mcds_cluster.remove_chromosome(exclude_chromosome)

In [ ]:
mcds_cluster.add_mc_frac(normalize_per_cell=True, clip_norm_value=10)


In [ ]:
bulk_dir = f'{ENTEX_ROOT}/bulk_mC/Loyfer2023/'
data_bulk = pd.read_hdf(f'{bulk_dir}Loyfer2023_1kb.hdf', key='data')


In [ ]:
selb = mcds_cluster.get_index('chrom1k').intersection(data_bulk.columns)
# selb = mcds_cluster.get_index('chrom1k').intersection(mcds.get_index('chrom1k'))
print(len(selb))

In [ ]:
mcds_cluster = mcds_cluster.sel({'chrom1k':selb})
data_cluster = mcds_cluster['chrom1k_da_frac'].to_pandas()

# mcds = mcds.sel({'chrom1k':selb})
# data_bulk = mcds['chrom1k_da_frac'].to_pandas()
data_bulk = data_bulk[selb]


In [ ]:
from sklearn.metrics import pairwise_distances
dist = pairwise_distances(data_bulk, data_cluster, 'cosine')
dist = pd.DataFrame(dist, columns=data_cluster.index, index=data_bulk.index)
dist.to_hdf(f'{outdir}cosine_subtype_Loyfer.hdf', key='data')


In [ ]:
dist = pd.read_hdf(f'{outdir}cosine_subtype_Loyfer.hdf', key='data')
sns.clustermap(dist, vmax=0.015, figsize=(20,20), xticklabels=2, yticklabels=2)


In [ ]:
dist = pd.read_hdf(f'{outdir}cosine_subtype_Loyfer.hdf', key='data').iloc[corder]
dist.columns = dist.columns.map(L2_annot)
bestbulk = np.argmin(dist, axis=0)
bestbulk[dist.min(axis=0)>0.01] = dist.shape[0]
dist = dist.iloc[:, np.argsort(bestbulk)]
# sns.clustermap(dist, vmax=0.015, row_cluster=False, col_cluster=False, cmap='cividis',
#                figsize=(20,20), xticklabels=2, yticklabels=2)


In [ ]:
fig, ax = plt.subplots(figsize=(12,12), dpi=300)
ax.imshow(dist, vmin=0, vmax=0.01, cmap='cividis')
xticks = np.arange(0, dist.shape[1], 3)
ax.set_xticks(xticks)
ax.set_xticklabels(dist.columns[xticks], rotation=90)
yticks = np.arange(0, dist.shape[0], 3)
ax.set_yticks(yticks)
ax.set_yticklabels(['-'.join(xx.split('-')[:-1]) for xx in dist.index[yticks]], rotation=0)
fig.savefig(f'{outdir}cosine_subtype_Loyfer.pdf', transparent=True)


## Schultz2015

In [ ]:
bulk_dir = f'{ENTEX_ROOT}/bulk_mC/Schultz2015/'


In [ ]:
mcds = MCDS.open(f'{bulk_dir}Schultz2015_1kb.mcds', var_dim='chrom1k')
mcds

In [ ]:
mcds = mcds.assign_coords({'chrom1k_chrom': ('chrom1k', mcds.coords['chrom1k_chrom'].data),
                    'chrom1k_start': ('chrom1k', mcds.coords['chrom1k_start'].data),
                    'chrom1k_end': ('chrom1k', mcds.coords['chrom1k_end'].data),
                   })
mcds

In [ ]:
# mcds.add_feature_cov_mean(var_dim='chrom1k', plot=False)
cov = mcds['chrom1k_da'].sel(count_type='cov').mean(dim='cell').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, binrange=(0,1200), ax=ax)

In [ ]:
mcds = mcds.sel({'chrom1k':cov.index[cov>20]})

In [ ]:
black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)

exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)

In [ ]:
mcds.add_mc_frac(normalize_per_cell=True, clip_norm_value=10)


In [ ]:
data = mcds['chrom1k_da_frac'].to_pandas()


In [ ]:
data.to_hdf(f'{bulk_dir}Schultz2015_1kb_hg38.hdf', key='data')


In [ ]:
data = pd.read_hdf(f'{bulk_dir}Schultz2015_1kb.hdf', key='data')


In [ ]:
from sklearn.metrics import pairwise_distances
dist = pairwise_distances(data, data, 'cosine')
dist = pd.DataFrame(dist, columns=data.index, index=data.index)
dist.to_hdf(f'{bulk_dir}cosine_Schultz2015.hdf', key='data')


In [ ]:
dist = pd.read_hdf(f'{bulk_dir}cosine_Schultz2015.hdf', key='data')
cg = sns.clustermap(dist, vmax=0.01, figsize=(10,10), xticklabels=1, yticklabels=1)
corder = cg.dendrogram_col.reordered_ind.copy()


In [ ]:
mcds_cluster = MCDS.open(f'{indir}merged_allc/subtype1k.mcds', var_dim='chrom1k')
mcds_cluster

In [ ]:
# group_meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', sep='\t', header=0, index_col=0)
# mcds_cluster = mcds_cluster.assign_coords(L2_any=('cell', group_meta.loc[mcds_cluster.get_index('cell'), 'L2_any']))
# mcds_cluster = mcds_cluster.groupby('L2_any').sum()
# mcds_cluster = MCDS(mcds_cluster, obs_dim='L2_any', var_dim='chrom1k')
mcds_cluster = mcds_cluster.sel({'mc_type':'CGN'})


In [ ]:
cov = mcds_cluster['chrom1k_da'].sel(count_type='cov').mean(dim='cell').squeeze().to_pandas()


In [ ]:
mcds_cluster = mcds_cluster.sel({'chrom1k':cov.index[cov>20]})

In [ ]:
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'
mcds_cluster = mcds_cluster.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)

exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds_cluster = mcds_cluster.remove_chromosome(exclude_chromosome)

In [ ]:
mcds_cluster.add_mc_frac(normalize_per_cell=True, clip_norm_value=10)


In [ ]:
bulk_dir = f'{ENTEX_ROOT}/bulk_mC/Schultz2015/'
data_bulk = pd.read_hdf(f'{bulk_dir}Schultz2015_1kb_hg38.hdf', key='data')


In [ ]:
selb = mcds_cluster.get_index('chrom1k').intersection(data_bulk.columns)
# selb = mcds_cluster.get_index('chrom1k').intersection(mcds.get_index('chrom1k'))
print(len(selb))

In [ ]:
mcds_cluster = mcds_cluster.sel({'chrom1k':selb})
data_cluster = mcds_cluster['chrom1k_da_frac'].to_pandas()

# mcds = mcds.sel({'chrom1k':selb})
# data_bulk = mcds['chrom1k_da_frac'].to_pandas()
data_bulk = data_bulk[selb]


In [ ]:
from sklearn.metrics import pairwise_distances
dist = pairwise_distances(data_bulk, data_cluster, 'cosine')
dist = pd.DataFrame(dist, columns=data_cluster.index, index=data_bulk.index)
dist.to_hdf(f'{outdir}cosine_subtype_Schultz.hdf', key='data')


In [ ]:
# dist = pd.read_hdf(f'{bulk_dir}cosine_L2any_Schultz.hdf', key='data')
# sns.clustermap(dist, figsize=(20,8), xticklabels=4, yticklabels=1)


In [ ]:
dist = pd.read_hdf(f'{bulk_dir}cosine_L2any_Schultz.hdf', key='data')
sns.clustermap(dist, vmax=0.012, figsize=(20,8), xticklabels=4, yticklabels=1)


In [ ]:
dist = pd.read_hdf(f'{outdir}cosine_subtype_Schultz.hdf', key='data')
cg = sns.clustermap(dist, vmax=0.012, figsize=(20,8), xticklabels=2, yticklabels=1)
rorder = cg.dendrogram_row.reordered_ind.copy()
dist = dist.iloc[rorder]

In [ ]:
dist = pd.read_hdf(f'{outdir}cosine_subtype_Schultz.hdf', key='data').iloc[corder]
dist.columns = dist.columns.map(L2_annot)
bestbulk = np.argmin(dist, axis=0)
bestbulk[dist.min(axis=0)>0.01] = dist.shape[0]
dist = dist.iloc[:, np.argsort(bestbulk)]
# bestbulk = np.argsort(np.arange(dist.shape[0])[None,:].dot(1-dist)[0])
# dist = dist.iloc[:, bestbulk]


In [ ]:
abbr_to_tissue = {
    "TH": "Thymus",
    "LG": "Lung",
    "RA": "Right atrium",
    "RV": "Right ventricle",
    "LV": "Left ventricle",
    "AO": "Aorta",
    "EG": "Oesophagus",
    "GA": "Gastric",
    "AD": "Adrenal gland",
    "PA": "Pancreas",
    "SX": "Spleen",
    "SB": "Small bowel",
    "SG": "Sigmoid colon",
    "FT": "Fat",
    "BL": "Bladder",
    "PO": "Psoas muscle",
    "OV": "Ovary",
    "LI": "Liver"
}


In [ ]:
fig, ax = plt.subplots(figsize=(10,5), dpi=300)
ax.imshow(dist, cmap='cividis', aspect='auto', rasterized=True, vmin=0, vmax=0.01)
xticks = np.arange(0, dist.shape[1], 3)
ax.set_xticks(xticks)
ax.set_xticklabels(dist.columns[xticks], rotation=90)
yticks = np.arange(0, dist.shape[0], 1)
ax.set_yticks(yticks)
ax.set_yticklabels(dist.index.str.split('_').str[0].map(abbr_to_tissue), rotation=0)
fig.savefig(f'{outdir}cosine_subtype_Schultz.pdf', transparent=True)
